In [3]:
import pandas as pd

metadata = pd.read_csv("../data/metadata.csv")

#Anzahl der Standorte
standorte = metadata["site_id"].nunique()
print(standorte)

19


In [4]:
# Gebäude pro Standort zählen
gebaeude_pro_standort = (
    metadata.groupby("site_id")["building_id"]
    .nunique()
    .sort_values(ascending=False)
)

print(gebaeude_pro_standort)

site_id
Rat         305
Hog         163
Lamb        147
Fox         137
Panther     136
Bull        124
Cockatoo    124
Eagle       106
Bear         92
Gator        74
Robin        52
Peacock      47
Bobcat       36
Wolf         36
Swan         21
Moose        15
Shrew         9
Mouse         7
Crow          5
Name: building_id, dtype: int64


In [7]:
#Gebäudenutzungstypen pro Standort zählen
gebaeudenutzung_pro_standort = (metadata.groupby("site_id")["primaryspaceusage"]
    .nunique().sort_values(ascending=False))

print(gebaeudenutzung_pro_standort)

site_id
Hog         13
Fox         12
Rat         11
Cockatoo    10
Lamb        10
Bobcat       8
Bear         7
Panther      7
Eagle        7
Bull         6
Wolf         6
Robin        5
Peacock      5
Gator        5
Mouse        3
Moose        1
Crow         1
Shrew        1
Swan         0
Name: primaryspaceusage, dtype: int64


In [8]:
#Prozentsatz der Gebäude mit fehlender Fläche (sqm) pro Standort
fehlende_flaeche_pro_standort = (metadata.groupby("site_id")["sqm"]
    .apply(lambda x: x.isna().mean()))

print(fehlende_flaeche_pro_standort)

site_id
Bear        0.0
Bobcat      0.0
Bull        0.0
Cockatoo    0.0
Crow        0.0
Eagle       0.0
Fox         0.0
Gator       0.0
Hog         0.0
Lamb        0.0
Moose       0.0
Mouse       0.0
Panther     0.0
Peacock     0.0
Rat         0.0
Robin       0.0
Shrew       0.0
Swan        0.0
Wolf        0.0
Name: sqm, dtype: float64


In [ ]:
#Wir brauchen hier nur die Spaltennamen, daher nrows=0, um nur die Spaltennamen zu bekommen
electricity = pd.read_csv(
    "../data/electricity.csv",
    nrows=0
)

#spaltennamen der electricity.csv
print(electricity.columns)

display(metadata[metadata["building_id"].isin(electricity.columns)][["building_id"]])

Index(['timestamp', 'Panther_parking_Lorriane', 'Panther_lodging_Cora',
       'Panther_office_Hannah', 'Panther_lodging_Hattie',
       'Panther_education_Teofila', 'Panther_education_Jerome',
       'Panther_retail_Felix', 'Panther_parking_Asia',
       'Panther_education_Misty',
       ...
       'Cockatoo_public_Caleb', 'Cockatoo_education_Tyler',
       'Cockatoo_public_Shad', 'Mouse_health_Buddy', 'Mouse_health_Modesto',
       'Mouse_lodging_Vicente', 'Mouse_health_Justin', 'Mouse_health_Ileana',
       'Mouse_health_Estela', 'Mouse_science_Micheal'],
      dtype='str', length=1579)


,building_id
3,Panther_education_Rosalie
4,Panther_education_Misty
6,Panther_education_Mattie
8,Panther_education_Diann
9,Panther_education_Gina
...,...
1631,Mouse_health_Estela
1632,Mouse_health_Ileana
1633,Mouse_health_Buddy
1634,Mouse_lodging_Vicente


In [34]:
metadata["hat_stromspalte"] = metadata["building_id"].isin(
    electricity.columns
)

display(
    metadata.groupby("site_id")[
        ["hat_stromspalte"]
    ].sum().sort_values("hat_stromspalte", ascending=False)
)

,hat_stromspalte
site_id,
Rat,305
Hog,152
Lamb,146
Fox,137
Bull,123
Cockatoo,117
Eagle,106
Panther,105
Bear,92


In [48]:
#pro Standort den Prozentsatz der Gebäude mit Stromspalte berechnen

stromabdeckung = (
    metadata.groupby("site_id")["hat_stromspalte"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(stromabdeckung.round(2))

site_id
Bear        100.00
Eagle       100.00
Crow        100.00
Wolf        100.00
Shrew       100.00
Fox         100.00
Gator       100.00
Robin       100.00
Mouse       100.00
Rat         100.00
Lamb         99.32
Bull         99.19
Bobcat       97.22
Peacock      95.74
Cockatoo     94.35
Hog          93.25
Swan         90.48
Moose        86.67
Panther      77.21
Name: hat_stromspalte, dtype: float64

In [47]:
flaechen = metadata[["sqm"]]

print((flaechen <= 0).sum())

sqm    0
dtype: int64


In [ ]:
#jetzt ganz einlesen die Elektricity.csv
electricity = pd.read_csv("../data/electricity.csv")

#Berechne für jede der 1.578 Stromspalten die prozentuale Vollständigkeit
electricity_with_no_timestamp = electricity.drop(columns=["timestamp"])

stromzaehler_vollstaendigkeit = (
    electricity_with_no_timestamp.notna().mean().mul(100).sort_values(ascending=False)
)

display(stromzaehler_vollstaendigkeit.round(2))

Lamb_health_Ken            100.00
Lamb_education_Camille     100.00
Lamb_office_Caitlin        100.00
Lamb_office_William        100.00
Lamb_office_Peggy          100.00
                            ...  
Rat_education_Mac            5.20
Bobcat_education_Seth        3.76
Bobcat_education_Barbra      0.63
Rat_public_Ulysses           0.00
Eagle_lodging_Garland        0.00
Length: 1578, dtype: float64

In [60]:
stromzaehler_standort_vollstaendigkeit = (
    metadata[["building_id", "site_id"]]
    .merge(
        stromzaehler_vollstaendigkeit.rename("vollstaendigkeit"),
        left_on="building_id",
        right_index=True,
    )
    .groupby("site_id")["vollstaendigkeit"]
    .mean()
    .sort_values(ascending=False)
)

display(stromzaehler_standort_vollstaendigkeit.round(2))

site_id
Shrew       100.00
Wolf         99.91
Hog          99.89
Mouse        99.81
Bull         99.79
Lamb         99.65
Fox          99.16
Panther      99.09
Robin        99.04
Eagle        97.75
Bear         96.39
Moose        95.63
Rat          92.42
Crow         92.15
Gator        90.95
Cockatoo     90.83
Peacock      87.58
Bobcat       74.62
Swan         49.87
Name: vollstaendigkeit, dtype: float64